## Download ACS 5-Year Data for Virginia Census Tracts

Downloads 2016 ACS 5-Year estimates at the census tract level for Virginia used for matching

Variables downloaded:
- Population
- Median household income
- Housing units
- Race
- Educational attainment

In [ ]:
import requests
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = next(
    p / "data" for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data").exists()
)

API_KEY = "Inserted Here"  
BASE_URL = "https://api.census.gov/data/2016/acs/acs5"
STATE = "51"  # Virginia FIPS code

In [18]:
# Variable definitions
# The Census API has a 50-variable limit per request, so we split into groups

variables = {
    # Population
    "B01001_001E": "total_population",

    # Median household income
    "B19013_001E": "median_household_income",

    # Housing units
    "B25001_001E": "total_housing_units",

    # Race (B02001)
    "B02001_001E": "race_total",
    "B02001_002E": "race_white_alone",

    # Educational attainment 25+ (B15003)
    "B15003_001E": "edu_total_25plus",
    "B15003_002E": "edu_no_schooling",
    "B15003_017E": "edu_high_school_diploma",
    "B15003_018E": "edu_ged",
    "B15003_019E": "edu_some_college_less_1yr",
    "B15003_020E": "edu_some_college_1yr_plus",
    "B15003_021E": "edu_associates",
    "B15003_022E": "edu_bachelors",
    "B15003_023E": "edu_masters",
    "B15003_024E": "edu_professional",
    "B15003_025E": "edu_doctorate",
}

print(f"Total variables to download: {len(variables)}")

Total variables to download: 16


In [ ]:
# Define variable groups
def fetch_acs(variable_codes, state, api_key):
    """
    Fetch ACS 5-year variables for all census tracts in a state.
    Returns a DataFrame.
    """
    # Always include NAME for tract identification
    get_vars = "NAME," + ",".join(variable_codes)

    params = {
        "get": get_vars,
        "for": "tract:*",
        "in": f"state:{state}",
        "key": api_key,
    }

    r = requests.get(BASE_URL, params=params)
    if r.status_code != 200:
        print(f"Error {r.status_code}: {r.text[:300]}")
        return pd.DataFrame()

    data = r.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    return df

In [19]:
# Fetch all variables (split into batches of 45 to stay under the 50-variable limit)
var_codes = list(variables.keys())
batch_size = 45
batches = [var_codes[i:i + batch_size] for i in range(0, len(var_codes), batch_size)]

dfs = []
for i, batch in enumerate(batches):
    print(f"Fetching batch {i+1}/{len(batches)}...")
    df = fetch_acs(batch, STATE, API_KEY)
    dfs.append(df)

# Merge all batches on state, county, tract
result = dfs[0]
for df in dfs[1:]:
    result = result.merge(df, on=["NAME", "state", "county", "tract"], how="outer")

print(f"\nShape: {result.shape}")
result.head()

Fetching batch 1/1...

Shape: (1907, 20)


,NAME,B01001_001E,B19013_001E,B25001_001E,B02001_001E,B02001_002E,B15003_001E,B15003_002E,B15003_017E,B15003_018E,B15003_019E,B15003_020E,B15003_021E,B15003_022E,B15003_023E,B15003_024E,B15003_025E,state,county,tract
0,"Census Tract 458.03, Virginia Beach city, Virg...",3052,70375,1518,3052,2051,2168,37,255,64,211,377,237,639,232,37,0,51,810,045803
1,"Census Tract 105.03, Stafford County, Virginia",2120,93750,829,2120,1938,1475,13,322,74,110,177,116,330,138,37,47,51,179,010503
2,"Census Tract 103.01, Stafford County, Virginia",4307,117100,1374,4307,3346,2950,32,701,231,176,425,217,579,290,33,0,51,179,010301
3,"Census Tract 200.02, Chesapeake city, Virginia",4514,47287,1834,4514,2555,2827,35,847,185,288,534,118,231,91,52,0,51,550,020002
4,"Census Tract 214.03, Chesapeake city, Virginia",4745,46061,1905,4745,2986,3060,0,868,120,228,503,380,288,104,0,22,51,550,021403


In [20]:
# Rename columns and create GEOID for joining to census shapefile
result = result.rename(columns=variables)
result["GEOID"] = result["state"] + result["county"] + result["tract"]

# Convert numeric columns from string to numeric
numeric_cols = list(variables.values())
result[numeric_cols] = result[numeric_cols].apply(pd.to_numeric, errors="coerce")

# Replace negative values with NaN  
result[numeric_cols] = result[numeric_cols].apply(lambda x: x.where(x >= 0, np.nan)) 

# Calculate percentage of white residents
result["pct_white"] = result["race_white_alone"] / result["race_total"] * 100

# Calculate percentage of adults with at least a high school diploma
edu_hs_or_above = (
    result["edu_high_school_diploma"] +
    result["edu_ged"] +
    result["edu_some_college_less_1yr"] +
    result["edu_some_college_1yr_plus"] +
    result["edu_associates"] +
    result["edu_bachelors"] +
    result["edu_masters"] +
    result["edu_professional"] +
    result["edu_doctorate"]
)
result["pct_hs_or_above"] = edu_hs_or_above / result["edu_total_25plus"] * 100

# Reorder columns — put calculated percentages after GEOID/NAME
result = result[["GEOID", "NAME", "state", "county", "tract", "pct_white", "pct_hs_or_above"] + numeric_cols]

print(f"Tracts: {len(result):,}")
print(f"Missing median income: {result['median_household_income'].isna().sum()}")
result.head()

Tracts: 1,907
Missing median income: 41


,GEOID,NAME,state,county,tract,pct_white,pct_hs_or_above,total_population,median_household_income,total_housing_units,...,edu_no_schooling,edu_high_school_diploma,edu_ged,edu_some_college_less_1yr,edu_some_college_1yr_plus,edu_associates,edu_bachelors,edu_masters,edu_professional,edu_doctorate
0,51810045803,"Census Tract 458.03, Virginia Beach city, Virg...",51,810,045803,67.201835,94.649446,3052,70375.0,1518,...,37,255,64,211,377,237,639,232,37,0
1,51179010503,"Census Tract 105.03, Stafford County, Virginia",51,179,010503,91.415094,91.593220,2120,93750.0,829,...,13,322,74,110,177,116,330,138,37,47
2,51179010301,"Census Tract 103.01, Stafford County, Virginia",51,179,010301,77.687485,89.898305,4307,117100.0,1374,...,32,701,231,176,425,217,579,290,33,0
3,51550020002,"Census Tract 200.02, Chesapeake city, Virginia",51,550,020002,56.601684,82.985497,4514,47287.0,1834,...,35,847,185,288,534,118,231,91,52,0
4,51550021403,"Census Tract 214.03, Chesapeake city, Virginia",51,550,021403,62.929399,82.124183,4745,46061.0,1905,...,0,868,120,228,503,380,288,104,0,22


In [21]:
#Remova NAs
result_clean = result.dropna(subset=numeric_cols)
print(f"Rows after dropping NaN: {len(result_clean)}")    

Rows after dropping NaN: 1866


In [22]:
#Filter only relevant columns
result_filtered = result_clean[[
    "GEOID", "NAME", "state", "county", "tract",                                                                                                                                         
    "total_population", "median_household_income", "total_housing_units",                                                                                                                
    "pct_white", "pct_hs_or_above"                                                                                                                                                       
]]                                                                                                                                                                                       
                
print(result_filtered.shape)                                                                                                                                                                 
result_filtered.head()

(1866, 10)


,GEOID,NAME,state,county,tract,total_population,median_household_income,total_housing_units,pct_white,pct_hs_or_above
0,51810045803,"Census Tract 458.03, Virginia Beach city, Virg...",51,810,045803,3052,70375.0,1518,67.201835,94.649446
1,51179010503,"Census Tract 105.03, Stafford County, Virginia",51,179,010503,2120,93750.0,829,91.415094,91.593220
2,51179010301,"Census Tract 103.01, Stafford County, Virginia",51,179,010301,4307,117100.0,1374,77.687485,89.898305
3,51550020002,"Census Tract 200.02, Chesapeake city, Virginia",51,550,020002,4514,47287.0,1834,56.601684,82.985497
4,51550021403,"Census Tract 214.03, Chesapeake city, Virginia",51,550,021403,4745,46061.0,1905,62.929399,82.124183


In [24]:
# Save
out_path = DATA_DIR / "raw" / "census_data" / "va_acs_2016_tract.csv"
result_filtered.to_csv(out_path, index=False)
print(f"Saved {len(result_filtered):,} rows to va_acs_2016_tract.csv")

Saved 1,866 rows to va_acs_2016_tract.csv
